# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIRˆ2 clinical dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All data elements are referenced by their Croissant schema `@id` for reproducibility.

### Dataset Source
This dataset is published under the [Open Data Commons BY 1.0 license](https://opendatacommons.org/licenses/by/1-0/) and is accessible via its Croissant schema at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema JSON-LD file)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List all available record sets and fields with their IDs. This section ensures that each dataset entity is referenced by its Croissant `@id`.

In [ ]:
# List all record sets and their fields using Croissant @ids:
from pprint import pprint

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        print(f"Record set: {rs['@id']}")
        fields = rs.get('fields', [])
        for fld in fields:
            print(f"  Field: {fld['@id']}")
        print()
else:
    # fallback: try to parse from dataset description
    print('No explicit record sets found in metadata. Attempting to access hidden structure.')
    # Try to find recordset ids from croissant Dataset interface itself
    record_sets = list(dataset.record_sets())
    for rs in record_sets:
        print(f"Record set: {rs['@id']}")
        print(f"  Fields:")
        for fld in rs['fields']:
            print(f"    {fld['@id']}")

## 3. Data Extraction
Load data from the primary record set into a DataFrame using its Croissant `@id` and the required field `@id`s.

**Note:** For this dataset, only one main record set is published, containing the clinical and biomarker data. We reference its `@id` and dynamically extract all fields.

In [ ]:
# Identify the available record set(s)
record_sets = list(dataset.record_sets())  # List of record sets (as dictionaries)
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Available record set @ids: {record_set_ids}")

# We'll use the first record set for extraction (update here if you wish to use a different one)
primary_recordset_id = record_set_ids[0]

# Load the records from the primary record set into a DataFrame
records = list(dataset.records(record_set=primary_recordset_id))
df = pd.DataFrame(records)

print(f"Available fields (@id) in '{primary_recordset_id}':")
for col in df.columns:
    print(f" - {col}")

df.head()

## 4. Exploratory Data Analysis (EDA)
Let's work with a numeric field in the dataset, identified by its `@id`. We'll demonstrate filtering, normalization, and grouping by categorical field—again, referencing all entities by their Croissant schema `@id`.

In [ ]:
# Find a numeric field in the DataFrame; here we select the first one automatically
import numpy as np

numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or np.issubdtype(df[col].dtype, np.number)]

if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Using numeric field '@id': {numeric_field_id}")
else:
    # If numeric fields are not detected, try to list all columns for manual selection
    print("No numeric fields detected! Available columns:")
    print(df.dtypes)

# Example: Work with a known numeric field if present (e.g. 'cr:field/age'), else skip EDA
if numeric_field_candidates:
    threshold = df[numeric_field_id].quantile(0.25)  # For demonstration, filter above 25th percentile
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize numeric column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized values for '@id': {numeric_field_id}")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical/nominal field if available
    group_field_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) or df[col].dtype.name == 'category']
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Grouping by field '@id': {group_field}")
        # Only group if the column is not too unique
        group_counts = filtered_df[group_field].nunique()
        if group_counts < 20:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Mean of {numeric_field_id} grouped by {group_field}:")
            print(grouped_df.head())
        else:
            print(f"Field '{group_field}' has too many unique groups for grouping display.")
    else:
        print("No suitable categorical fields found for grouping.")
else:
    print("No EDA performed as no numeric fields could be detected.")

## 5. Visualization
Visualize the distribution for the selected numeric field, and if grouping was successful, display group comparisons. All axes and figure annotations are referenced by their Croissant `@id`.

We'll use matplotlib for basic visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.xlabel(f"{numeric_field_id}")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No visualization due to lack of numeric data.")

## 6. Conclusion
This notebook demonstrated how to programmatically explore, filter, normalize, and visualize clinical tabular data from the FAIRˆ2 dataset using the `mlcroissant` library. All data operations referenced dataset entities by their Croissant `@id`, maximizing reproducibility and semantic clarity.

**Key observations:**
- The FAIRˆ2 dataset exposes clinical fields and molecular biomarkers with rich provenance via its Croissant schema.
- Numeric and categorical fields can be dynamically identified for flexible downstream analysis.
- Visualizations and data pipelines become more robust by referencing all components by explicit `@id`.

For more, see the [Croissant documentation](https://mlcommons.github.io/croissant/) or [mlcroissant GitHub](https://github.com/mlcommons/croissant).